In [2]:
import cv2 as cv
import mediapipe as mp
import numpy as np
import time
import joblib
from datetime import datetime
import json

MODEL_PATH = 'Random_forest_focus_model.pkl'
CONFIDENCE_THRESHOLD = 0.60
CLASSES = {0: "Focused",1: "Head Turned",2: "Looking Down",3: "Eyes Distracted",4: "Drowsy"}

stats = {k: 0.0 for k in CLASSES.values()}
session_start = datetime.now()
try:
    model = joblib.load(MODEL_PATH)
    print("Model loaded")
except Exception as e:
    print(f"Error loading model: {e}")
    exit(1)
    
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)
cap = cv.VideoCapture(0)

def normalize_input(landmarks_list):
    landmarks = np.array(landmarks_list).reshape(-1, 3)
    nose = landmarks[4]
    landmarks -= nose
    dist = np.linalg.norm(landmarks[33]- landmarks[263])
    if dist != 0:
        landmarks /= dist
    return landmarks.flatten().reshape(1, -1)

print("Starting video capture. Press 'q' to quit.")

last_frame_time = time.time()
current_status = "Focused"

while cap.isOpened():
    success, frame = cap.read()
    
    if not success:
        break
    
    frame = cv.flip(frame, 1)
    rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)
    
    now = time.time()
    
    delta = now - last_frame_time
    
    if results.multi_face_landmarks and delta > 0.1:
        for face_landmarks in results.multi_face_landmarks:
            
            raw_landmarks = []
            for lm in face_landmarks.landmark:
                raw_landmarks.extend([lm.x, lm.y, lm.z])
            
            norm_features = normalize_input(raw_landmarks)
            
            probabilites = model.predict_proba(norm_features)[0]
            
            max_prob = np.max(probabilites)
            prediction_index = np.argmax(probabilites)
            
            if max_prob > CONFIDENCE_THRESHOLD:
                current_status = CLASSES[prediction_index]
            else:
                current_status = "Focused"
                
            stats[current_status] += delta
            
            color = (0, 255, 0) if current_status == "Focused" else (0, 0, 255)
            cv.putText(frame, f"{current_status} ({max_prob:.2f})", (10, 30), cv.FONT_HERSHEY_SIMPLEX, 1, color, 2)
            
    cv.imshow('AI productivity tracker', frame)
    if cv.waitKey(1) & 0xFF == ord('q'):
        break
    
cap.release()
cv.destroyAllWindows()
session_end = datetime.now()

total_time = sum(stats.values())
focused_time = stats["Focused"]
productivity_score = (focused_time / total_time) * 100 if total_time > 0 else 0


report = {
    "session_date":session_start.strftime("%Y-%m-%d %H:%M:%S"),
    "duration_minutes" : total_time / 60,
    "focus_percentage": productivity_score,
    "distracted_time_minutes": {k : round(v , 2) for k, v in stats.items() }
}

report_filename = f"report_{session_start.strftime('%Y%m%d_%H%M%S')}.json"
with open(report_filename, "w") as f:
    json.dump(report, f, indent=4)

print(f"\nSession Complete! Report saved to {report_filename}")
print(f"Final Productivity Score: {round(productivity_score, 2)}%")

Model loaded
Starting video capture. Press 'q' to quit.


c:\Users\User\Desktop\OpenCV\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\User\Desktop\OpenCV\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\User\Desktop\OpenCV\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
c:\Users\User\Desktop\


Session Complete! Report saved to report_20260421_065639.json
Final Productivity Score: 87.97%
